# 🔋 Battery Dataset Exploration

This notebook explores the **SP20** Li-ion battery datasets used in the CWT-CNN SOC estimation pipeline.

**Datasets:**
| Dataset | Profile | Temperature | Purpose |
|---------|---------|-------------|----------|
| DST @ 0°C | Dynamic Stress Test | 0°C | Training |
| DST @ 25°C | Dynamic Stress Test | 25°C | Testing |
| US06 @ 25°C | US06 Drive Cycle | 25°C | Testing |
| FUDS @ 25°C | Federal Urban Drive | 25°C | Testing |

**Pipeline:** Raw Signals → Coulomb Counting (SOC) → Sliding Windows → CWT Scalograms → CNN

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML

# Use inline backend for notebook
%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Project imports
from data_preprocessing.preprocess import load_signals, estimate_fs, compute_soc
from config.settings import (
    TRAIN_DATA, TEST_DATA,
    INITIAL_SOC, CAPACITY_AH,
    WINDOW_SIZE, STRIDE
)

print('✅ All imports successful')

## 2. Load All Datasets

We load the training dataset (DST @ 0°C) and all three test datasets (DST, US06, FUDS @ 25°C).

In [ ]:
# Combine all dataset configs into one list for easy iteration
ALL_DATASETS = [TRAIN_DATA] + TEST_DATA

# Storage for loaded data
loaded = {}

for cfg in ALL_DATASETS:
    label = cfg['label']
    print(f'\n── Loading: {label} ──')
    print(f'   File: {os.path.basename(cfg["path"])}')
    
    df, voltage, current, temperature, time = load_signals(
        cfg['path'],
        sheet_name=cfg['sheet'],
        ambient_temp=cfg['ambient_temp']
    )
    
    fs = estimate_fs(time)
    
    loaded[label] = {
        'df': df,
        'voltage': voltage,
        'current': current,
        'temperature': temperature,
        'time': time,
        'fs': fs,
        'config': cfg
    }
    
    print(f'   Samples : {len(voltage):,}')
    print(f'   Duration: {time[-1]/3600:.2f} hours ({time[-1]:.0f} s)')
    print(f'   Fs      : {fs:.4f} Hz')
    print(f'   V range : [{voltage.min():.3f}, {voltage.max():.3f}] V')
    print(f'   I range : [{current.min():.3f}, {current.max():.3f}] A')

print(f'\n✅ Loaded {len(loaded)} datasets')

## 3. Raw Data Preview

Quick look at the raw DataFrame structure for each dataset.

In [ ]:
for label, data in loaded.items():
    print(f'\n{"═"*60}')
    print(f'  {label}')
    print(f'{"═"*60}')
    print(f'Shape: {data["df"].shape}')
    print(f'Columns: {list(data["df"].columns)}')
    display(data['df'].head())
    print(f'\nDescriptive Statistics:')
    display(data['df'].describe())

## 4. Dataset Summary Statistics

In [ ]:
# Build a summary comparison table
summary_rows = []
for label, data in loaded.items():
    v, i, t = data['voltage'], data['current'], data['time']
    summary_rows.append({
        'Dataset': label,
        'Samples': len(v),
        'Duration (h)': f"{t[-1]/3600:.2f}",
        'Fs (Hz)': f"{data['fs']:.4f}",
        'V_min (V)': f"{v.min():.3f}",
        'V_max (V)': f"{v.max():.3f}",
        'V_mean (V)': f"{v.mean():.3f}",
        'I_min (A)': f"{i.min():.3f}",
        'I_max (A)': f"{i.max():.3f}",
        'I_mean (A)': f"{i.mean():.3f}",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.set_caption('📊 Dataset Summary').set_table_styles(
    [{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]
))

## 5. Voltage & Current Profiles

Visualize the raw voltage and current signals for all datasets side by side.

In [ ]:
fig, axes = plt.subplots(len(loaded), 2, figsize=(16, 4 * len(loaded)), sharex=False)

colors_v = ['#E53935', '#FF7043', '#F4511E', '#D84315']  # Red tones for voltage
colors_i = ['#1E88E5', '#42A5F5', '#2196F3', '#1565C0']  # Blue tones for current

for idx, (label, data) in enumerate(loaded.items()):
    t_h = data['time'] / 3600  # Convert to hours
    
    # Voltage subplot
    ax_v = axes[idx, 0]
    ax_v.plot(t_h, data['voltage'], color=colors_v[idx], linewidth=0.6, alpha=0.9)
    ax_v.set_ylabel('Voltage (V)', fontsize=11)
    ax_v.set_title(f'{label} — Voltage', fontsize=12, fontweight='bold')
    ax_v.fill_between(t_h, data['voltage'].min(), data['voltage'], alpha=0.08, color=colors_v[idx])
    
    # Current subplot
    ax_i = axes[idx, 1]
    ax_i.plot(t_h, data['current'], color=colors_i[idx], linewidth=0.6, alpha=0.9)
    ax_i.set_ylabel('Current (A)', fontsize=11)
    ax_i.set_title(f'{label} — Current', fontsize=12, fontweight='bold')
    ax_i.fill_between(t_h, 0, data['current'], alpha=0.08, color=colors_i[idx])
    ax_i.axhline(y=0, color='gray', linestyle='--', linewidth=0.5, alpha=0.6)

# Common x-label
for ax in axes[-1]:
    ax.set_xlabel('Time (hours)', fontsize=11)

fig.suptitle('Raw Voltage & Current Profiles — All Datasets', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Voltage & Current Overlay (Test Datasets)

Compare the three test drive profiles (DST, US06, FUDS @ 25°C) overlaid on the same axes.

In [ ]:
test_labels = [cfg['label'] for cfg in TEST_DATA]
test_colors = {'DST @ 25°C': '#1E88E5', 'US06 @ 25°C': '#43A047', 'FUDS @ 25°C': '#E53935'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8))

for label in test_labels:
    data = loaded[label]
    t_s = data['time']  # Keep in seconds for overlay
    c = test_colors.get(label, '#333')
    ax1.plot(t_s, data['voltage'], color=c, linewidth=0.5, alpha=0.8, label=label)
    ax2.plot(t_s, data['current'], color=c, linewidth=0.5, alpha=0.8, label=label)

ax1.set_ylabel('Voltage (V)', fontsize=12)
ax1.set_title('Voltage Comparison — Test Datasets @ 25°C', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)

ax2.set_ylabel('Current (A)', fontsize=12)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_title('Current Comparison — Test Datasets @ 25°C', fontsize=13, fontweight='bold')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 7. Current Distribution Analysis

Histograms of current for each drive profile to understand charge/discharge patterns.

In [ ]:
fig, axes = plt.subplots(1, len(loaded), figsize=(16, 4))

hist_colors = ['#7E57C2', '#26A69A', '#FF7043', '#5C6BC0']

for idx, (label, data) in enumerate(loaded.items()):
    ax = axes[idx]
    ax.hist(data['current'], bins=100, color=hist_colors[idx], alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Current (A)')
    ax.set_ylabel('Count')
    
    # Annotate charge/discharge ratio
    charge_pct = (data['current'] > 0).sum() / len(data['current']) * 100
    discharge_pct = (data['current'] < 0).sum() / len(data['current']) * 100
    ax.text(0.95, 0.95, f'Charge: {charge_pct:.1f}%\nDischarge: {discharge_pct:.1f}%',
            transform=ax.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Current Distribution — Charge vs Discharge', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. SOC Calculation (Coulomb Counting)

Compute the State of Charge using the Coulomb Counting method:

$$SOC(t) = SOC_0 + \frac{1}{Q_{rated}} \int_0^t I(\tau) \, d\tau$$

**Parameters:**
- Initial SOC: `0.8` (80%)
- Rated capacity: `2.0 Ah` (SP20 cell)

In [ ]:
print(f'SOC Computation Parameters:')
print(f'  Initial SOC  : {INITIAL_SOC} ({INITIAL_SOC*100:.0f}%)')
print(f'  Capacity     : {CAPACITY_AH} Ah')
print(f'  Window Size  : {WINDOW_SIZE} samples')
print(f'  Stride       : {STRIDE} samples')
print()

# Compute SOC for all datasets
for label, data in loaded.items():
    soc = compute_soc(
        data['current'], data['time'],
        initial_soc=INITIAL_SOC,
        capacity_ah=CAPACITY_AH
    )
    data['soc'] = soc  # Store for later use
    
    print(f'{label}:')
    print(f'  SOC range: [{soc.min():.4f}, {soc.max():.4f}]')
    print(f'  SOC start: {soc[0]:.4f} → SOC end: {soc[-1]:.4f}')
    print(f'  Δ SOC    : {soc[0] - soc[-1]:.4f} ({(soc[0] - soc[-1])*100:.2f}%)')
    print()

## 9. SOC Profiles — Individual Plots

Visualize how SOC decays over time for each drive profile.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

soc_colors = ['#AB47BC', '#26A69A', '#EF5350', '#5C6BC0']
fill_colors = ['#CE93D8', '#80CBC4', '#EF9A9A', '#9FA8DA']

for idx, (label, data) in enumerate(loaded.items()):
    ax = axes[idx]
    t_h = data['time'] / 3600
    soc_pct = data['soc'] * 100
    
    ax.plot(t_h, soc_pct, color=soc_colors[idx], linewidth=1.8)
    ax.fill_between(t_h, 0, soc_pct, alpha=0.15, color=fill_colors[idx])
    
    ax.set_title(f'SOC — {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (hours)', fontsize=11)
    ax.set_ylabel('SOC (%)', fontsize=11)
    ax.set_ylim(-2, 85)
    
    # Annotate start/end
    ax.annotate(f'{soc_pct[0]:.1f}%', xy=(t_h[0], soc_pct[0]),
                fontsize=10, fontweight='bold', color=soc_colors[idx],
                xytext=(10, 5), textcoords='offset points')
    ax.annotate(f'{soc_pct[-1]:.1f}%', xy=(t_h[-1], soc_pct[-1]),
                fontsize=10, fontweight='bold', color=soc_colors[idx],
                xytext=(-40, 5), textcoords='offset points')

fig.suptitle('State of Charge (Coulomb Counting) — All Datasets',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10. SOC Overlay Comparison

All four SOC profiles overlaid for direct comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

overlay_colors = {
    'DST @ 0°C': '#AB47BC',
    'DST @ 25°C': '#1E88E5',
    'US06 @ 25°C': '#43A047',
    'FUDS @ 25°C': '#E53935'
}

for label, data in loaded.items():
    t_h = data['time'] / 3600
    c = overlay_colors.get(label, '#333')
    ax.plot(t_h, data['soc'] * 100, color=c, linewidth=2, label=label, alpha=0.85)

ax.set_xlabel('Time (hours)', fontsize=13)
ax.set_ylabel('SOC (%)', fontsize=13)
ax.set_title('SOC Comparison — All Drive Profiles', fontsize=15, fontweight='bold')
ax.set_ylim(-2, 85)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.show()

## 11. Combined View: Voltage, Current & SOC

Three-panel view for each dataset showing how V, I, and SOC evolve together.

In [ ]:
for label, data in loaded.items():
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
    t_h = data['time'] / 3600
    
    # Voltage
    ax1.plot(t_h, data['voltage'], color='#E53935', linewidth=0.7)
    ax1.set_ylabel('Voltage (V)', fontsize=11)
    ax1.set_title(f'{label} — Voltage / Current / SOC', fontsize=13, fontweight='bold')
    
    # Current
    ax2.plot(t_h, data['current'], color='#1E88E5', linewidth=0.7)
    ax2.set_ylabel('Current (A)', fontsize=11)
    ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    
    # SOC
    ax3.plot(t_h, data['soc'] * 100, color='#43A047', linewidth=1.5)
    ax3.fill_between(t_h, 0, data['soc'] * 100, alpha=0.1, color='#43A047')
    ax3.set_ylabel('SOC (%)', fontsize=11)
    ax3.set_xlabel('Time (hours)', fontsize=11)
    ax3.set_ylim(-2, 85)
    
    plt.tight_layout()
    plt.show()
    print()

## 12. Zoomed View — First 500 Seconds

Close-up look at the initial cycling pattern to understand the dynamic load profile structure.

In [ ]:
zoom_seconds = 500  # First 500 seconds

for label, data in loaded.items():
    mask = data['time'] <= zoom_seconds
    t = data['time'][mask]
    v = data['voltage'][mask]
    i = data['current'][mask]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
    
    ax1.plot(t, v, color='#E53935', linewidth=1)
    ax1.set_ylabel('Voltage (V)', fontsize=11)
    ax1.set_title(f'{label} — First {zoom_seconds}s (Zoomed)', fontsize=12, fontweight='bold')
    
    ax2.plot(t, i, color='#1E88E5', linewidth=1)
    ax2.set_ylabel('Current (A)', fontsize=11)
    ax2.set_xlabel('Time (s)', fontsize=11)
    ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
    print()

## 13. Voltage vs Current — Scatter Analysis

V-I scatter plots reveal the battery's internal resistance characteristics and operating range.

In [ ]:
fig, axes = plt.subplots(1, len(loaded), figsize=(16, 4))

scatter_cmaps = ['Purples', 'Blues', 'Greens', 'Reds']

for idx, (label, data) in enumerate(loaded.items()):
    ax = axes[idx]
    # Downsample for faster rendering
    step = max(1, len(data['current']) // 5000)
    i_ds = data['current'][::step]
    v_ds = data['voltage'][::step]
    soc_ds = data['soc'][::step]
    
    sc = ax.scatter(i_ds, v_ds, c=soc_ds * 100, cmap=scatter_cmaps[idx],
                    s=3, alpha=0.5, edgecolors='none')
    ax.set_xlabel('Current (A)', fontsize=10)
    ax.set_ylabel('Voltage (V)', fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    plt.colorbar(sc, ax=ax, label='SOC (%)', shrink=0.8)

fig.suptitle('V-I Characteristic (colored by SOC)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 14. SOC Summary Table

In [ ]:
soc_summary = []
for label, data in loaded.items():
    soc = data['soc']
    soc_summary.append({
        'Dataset': label,
        'SOC Start (%)': f"{soc[0]*100:.2f}",
        'SOC End (%)': f"{soc[-1]*100:.2f}",
        'SOC Min (%)': f"{soc.min()*100:.2f}",
        'SOC Max (%)': f"{soc.max()*100:.2f}",
        'Δ SOC (%)': f"{(soc[0]-soc[-1])*100:.2f}",
        'Duration (h)': f"{data['time'][-1]/3600:.2f}",
    })

soc_df = pd.DataFrame(soc_summary)
display(soc_df.style.set_caption('🔋 SOC Summary — Coulomb Counting').set_table_styles(
    [{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]
))

## 15. Paper Figure 3 — US06 & FUDS Raw Signals

Recreating Figure 3 from the paper: 2×3 grid showing Current, Voltage, and SOC for US06 and FUDS.

In [ ]:
us06 = loaded['US06 @ 25°C']
fuds = loaded['FUDS @ 25°C']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

blue = '#3182bd'

# Top Row: US06
axes[0, 0].plot(us06['time'], us06['current'], color=blue, linewidth=0.5)
axes[0, 0].set_title('(a) US06 — Current', fontweight='bold')
axes[0, 0].set_ylabel('Current (A)')

axes[0, 1].plot(us06['time'], us06['voltage'], color=blue, linewidth=0.5)
axes[0, 1].set_title('(b) US06 — Voltage', fontweight='bold')
axes[0, 1].set_ylabel('Voltage (V)')

axes[0, 2].plot(us06['time'], us06['soc'] * 100, color=blue, linewidth=1.5)
axes[0, 2].set_title('(c) US06 — SOC', fontweight='bold')
axes[0, 2].set_ylabel('SOC (%)')
axes[0, 2].set_ylim(0, 85)

# Bottom Row: FUDS
axes[1, 0].plot(fuds['time'], fuds['current'], color=blue, linewidth=0.5)
axes[1, 0].set_title('(d) FUDS — Current', fontweight='bold')
axes[1, 0].set_ylabel('Current (A)')

axes[1, 1].plot(fuds['time'], fuds['voltage'], color=blue, linewidth=0.5)
axes[1, 1].set_title('(e) FUDS — Voltage', fontweight='bold')
axes[1, 1].set_ylabel('Voltage (V)')

axes[1, 2].plot(fuds['time'], fuds['soc'] * 100, color=blue, linewidth=1.5)
axes[1, 2].set_title('(f) FUDS — SOC', fontweight='bold')
axes[1, 2].set_ylabel('SOC (%)')
axes[1, 2].set_ylim(0, 85)

for ax in axes.flat:
    ax.set_xlabel('Time (s)')
    ax.grid(True, alpha=0.3)

fig.suptitle('Figure 3 — Raw Signals (US06 & FUDS @ 25°C)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## ✅ Summary

This notebook verified:

1. **Data Loading** — All 4 datasets load correctly from Arbin XLS format
2. **Voltage/Current Profiles** — Drive profiles (DST, US06, FUDS) show expected dynamic patterns
3. **SOC Calculation** — Coulomb Counting produces monotonically decreasing SOC from 80% initial
4. **SOC Range** — All datasets discharge from ~80% to near 0%, consistent with test protocol

**Next Steps:** Run `main.py` to generate CWT scalograms and train the CNN model.